# L1b: Time Value of Money, Abstract Assets and the Net Present Value
Today, we will introduce the concepts of _Abstract Assets_ and the _Time Value of Money_. These concepts are foundational to understanding finance and investment, and we will see them over and over again throughout the course.

> __Learning Objectives:__
> 
> By the end of this lecture, students should be able to define and demonstrate the following concepts:
> * __Time value of money__: The time value of money is the principle that otherwise identical cash flows delivered at different dates generally have different present values. Discounting converts dated cash flows to a common valuation date. The relevant rate reflects the chosen benchmark and may incorporate opportunity cost, inflation, liquidity, and risk; TVM is not simply a claim that nominal money mechanically loses value.
> * __Abstract Asset__: An abstract asset is a series of current and future cash flows. This framework can model the value of everything you consider an asset, such as cash, stocks, and bonds, or physical assets, such as a car or house.
> * __Net Present Value (NPV)__: Net present value is the sum of signed cash flows expressed at a common valuation date using a declared discount curve. Positive NPV means value in excess of that specific pricing benchmark under the stated cash-flow and risk assumptions; it does not by itself establish realized profit or outperformance of a risk-free investment.

Let's get started!

___

## Setup, Data, and Prerequisites
We set up the computational environment by including the `Include.jl` file, loading any needed resources, such as sample datasets, and setting up any required constants. 

> The `Include.jl` file also loads external packages, various functions that we will use in the exercise, and custom types to model the components of our problem. It checks for a `Manifest.toml` file. If it finds one, packages are loaded; otherwise, packages are downloaded and then loaded.

For clarity, `include(joinpath(@__DIR__, "Include.jl"))` loads the `Include.jl` file that lives next to this notebook. `@__DIR__` expands to the directory containing the notebook, so this call works even if you open the notebook from a different working directory in VSCode.

In [3]:
include(joinpath(@__DIR__, "Include.jl")); # this sets up the environment, we'll do this all the time, on everything we do

For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 Quantitative Finance Package documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

___

## Time Value of Money
The time value of money is the principle that otherwise identical cash flows delivered at different dates are different dated claims. Before they can be compared or added, they must be expressed at a common valuation date using a stated conversion rule.

Inflation, opportunity cost, uncertainty, liquidity, and preferences for timing can all affect that rule. Therefore, a later dollar is not automatically worth less under every possible model; its present value depends on the selected benchmark and assumptions.

Suppose we have $P$ USD at time $0$. Let $y$ denote a quoted nominal annual yield used for valuation. With one compounding interval per year, the amount after one year is
$$
\begin{align*}
F_1=\underbrace{(1+y)}_{\mathcal D_{1,0}(y;1)}P.
\end{align*}
$$
The factor $\mathcal D_{1,0}(y;1)>1$ moves time-0 dollars forward and is called a __forward accumulation factor__. Its inverse $\mathcal D_{1,0}^{-1}(y;1)<1$ moves future dollars back to time $0$ and is the corresponding __discount factor__.


## Discrete Compounding and Accumulation Factors
Suppose $j=0,1,2,\ldots$ is a discrete compounding-period index. Let $q_{\ell+1,\ell}>-1$ be the effective rate over interval $\ell\rightarrow\ell+1$. Repeated accumulation gives
$$
\begin{align*}
F_j
=\underbrace{\left[\prod_{\ell=0}^{j-1}(1+q_{\ell+1,\ell})\right]}_{\mathcal D_{j,0}(\mathbf q)}P,
\qquad j=0,1,2,\ldots
\end{align*}
$$
where the empty product at $j=0$ equals one. The product $\mathcal D_{j,0}(\mathbf q)$ is the forward accumulation factor from time $0$ to period $j$.

For the common constant-rate convention, let $y$ be a nominal annual yield and let the positive integer $n$ be the number of compounding intervals per year. The effective rate per interval is $y/n$, so
$$
\boxed{
\begin{align*}
\mathcal D_{j,0}(y;n)=\left(1+\frac{y}{n}\right)^j.
\end{align*}}
$$
Its inverse
$$
\mathcal D_{j,0}^{-1}(y;n)=\left(1+\frac{y}{n}\right)^{-j}
$$
discounts a period-$j$ cash flow to time $0$.

If a horizon of $T$ years lies on the compounding grid, then $j=nT$. We may therefore write the same factor using elapsed time:
$$
\boxed{
\begin{align*}
\mathcal D_{T,0}(y;n)
:=\left(1+\frac{y}{n}\right)^{nT}.
\end{align*}}
$$
For example, semiannual compounding has $n=2$; a three-year horizon corresponds to $j=nT=6$ compounding periods.


## Continuous Compounding
Let $g(t)$ denote a continuously compounded growth rate with units of inverse time. The forward accumulation factor over $0\rightarrow T$ is
$$
\begin{align*}
\mathcal D_{T,0}(g)
&=\exp\left(\int_0^T g(u)\,du\right).
\end{align*}
$$
If $g(t)=g$ is constant, then
$$
\boxed{
\mathcal D_{T,0}(g)=e^{gT},
\qquad
\mathcal D_{T,0}^{-1}(g)=e^{-gT}.
}
$$

A nominal annual yield $y$ compounded $n$ times per year has the continuously compounded equivalent
$$
\boxed{
g_y:=n\log\left(1+\frac{y}{n}\right).
}
$$
Consequently,
$$
\left(1+\frac{y}{n}\right)^{nT}=e^{g_yT}.
$$
These are two representations of the same accumulation rule. Separately, if the numerical annual-rate parameter $y$ is held fixed while $n\rightarrow\infty$, then
$$
\lim_{n\rightarrow\infty}\left(1+\frac{y}{n}\right)^{nT}=e^{yT}.
$$
This limiting statement does not mean that every quoted nominal yield is already a continuously compounded rate.

Discrete and continuous conventions can be used for many asset classes; the appropriate choice is determined by the contract, quotation convention, and valuation model.

> __Example:__ Invest 1 USD today for $T$ years. Compare annual compounding, semiannual compounding, and the continuous-compounding limit using the same numerical annual-rate parameter. How do the accumulation factors differ?

So what happens?


In [ ]:
let

    # initialize -
    discrete_compounding_model = DiscreteCompoundingModel();
    T = 30.0;  # horizon in years
    y = 0.10;  # numerical annual-rate parameter
    n₁ = 1;    # annual compounding
    n₂ = 2;    # semiannual compounding
    N₁ = Int(n₁*T); # total compounding periods in case 1
    N₂ = Int(n₂*T); # total compounding periods in case 2
    𝒟(g,t) = exp(g*t); # continuous accumulation factor

    # compute discrete accumulation factors using the package's discount(...) API
    accumulation_dictionary_case_1 = discount(discrete_compounding_model, y, N₁, λ = n₁);
    accumulation_dictionary_case_2 = discount(discrete_compounding_model, y, N₂, λ = n₂);

    # convert accumulation data to arrays -
    j₁ = keys(accumulation_dictionary_case_1) |> collect |> sort;
    X₁ = Array{Float64,2}(undef, length(j₁),2);
    for i ∈ eachindex(j₁)
        j = j₁[i]
        X₁[i,1] = j/n₁; # elapsed time in years
        X₁[i,2] = accumulation_dictionary_case_1[j];
    end

    j₂ = keys(accumulation_dictionary_case_2) |> collect |> sort;
    X₂ = Array{Float64,2}(undef, length(j₂),2);
    for i ∈ eachindex(j₂)
        j = j₂[i]
        X₂[i,1] = j/n₂; # elapsed time in years
        X₂[i,2] = accumulation_dictionary_case_2[j];
    end

    # continuous-compounding limit with g = y -
    t₃ = 0:0.001:T |> collect;
    X₃ = Array{Float64,2}(undef, length(t₃),2);
    for i ∈ eachindex(t₃)
        t = t₃[i]
        X₃[i,1] = t;
        X₃[i,2] = 𝒟(y,t);
    end

    plot(X₁[:,1], X₁[:,2], label = "Annual compounding", xlabel = "Years", ylabel = "Accumulation factor", legend = :bottomright, c=:navy, lw=2)
    plot!(X₂[:,1], X₂[:,2], label = "Semiannual compounding", c=:red, lw=2)
    plot!(X₃[:,1], X₃[:,2], label = "Continuous-compounding limit", c=:green, lw=2)
end

__Should we expect these values to be close?__
For a fixed numerical annual-rate parameter $y$,
$$
\mathcal D_{T,0}(y;n)=\left(1+\frac{y}{n}\right)^{nT}
$$
approaches $e^{yT}$ as $n$ increases. Expanding the logarithm gives
$$
\begin{align*}
\mathcal D_{T,0}(y;n)
&=e^{yT}\exp\!\left(-\frac{y^2T}{2n}+\frac{y^3T}{3n^2}-\cdots\right).
\end{align*}
$$
Thus, for large $n$, the relative gap is approximately
$$
\boxed{
\begin{align*}
1-\frac{\mathcal D_{T,0}(y;n)}{e^{yT}}
\approx \frac{y^2T}{2n}.
\end{align*}}
$$
This is a convergence comparison. For a fixed finite-$n$ nominal yield, using $g_y=n\log(1+y/n)$ instead gives exact equivalence rather than an approximation.

Let's check the convergence gap for selected values of $n$ and $y$:


In [ ]:
let
    # initialize -
    T = 30.0; # horizon in years
    y = 0.15; # numerical annual-rate parameter
    n = 1;    # compounding intervals per year

    gap = (y^2 * T) / (2 * n); # leading-order relative gap

    println("Approximate gap with n = $(n) over T = $(T) years at y = $(y): $(gap).")
end

## Who Sets the Valuation Rate?
A valuation rate is not universal; it depends on the cash flow, market convention, benchmark, horizon, and risks being modeled. Market rates and monetary policy help shape those inputs, but no single policy rate prices every asset.

### The Federal Reserve
The United States Federal Reserve, created by the [Federal Reserve Act of 1913](https://www.federalreserve.gov/aboutthefed/fract.htm), is the nation’s central banking system, comprising a Board of Governors in Washington, D.C., and twelve regional Federal Reserve Banks. 

> __The Federal Reserve__ conducts monetary policy, via tools such as open market operations, the discount rate, and reserve requirements, to promote maximum employment, stable prices, and moderate long-term interest rates, while also supervising banks and serving as the lender of last resort.
>
> __Federal Open Market Committee (FOMC)__ sets a target range for the federal funds rate, the overnight rate on unsecured reserve balances traded between eligible institutions. Banks borrow directly from Federal Reserve Banks through the discount window at separately administered rates. Other market rates respond to policy and market conditions but are not mechanically pegged to the federal funds rate.
>
> __Optional__: Meet Susan and Ronnie from the Federal Reserve — [watch here](https://youtu.be/xHQJBNO0yQc).

The FOMC's decisions influence market yields and other valuation inputs, affecting everything from mortgage rates to corporate borrowing costs. They also impact consumer and investor sentiment. Thus, the FOMC plays a crucial role in shaping the economic landscape.

What do current market-based measures and prediction markets imply about the path of monetary policy at the [next FOMC meeting?](https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm) Treat these probabilities as changing market estimates, not as policy commitments.

### What valuation rate should I use?
_It depends_ on risk. The higher the risk, the higher required return. Here are the rough ranges:

* **Safe stuff (2-5%)**: Treasury bills around 5%, AAA corporate bonds around 5%. Sleep well at night territory. Although these instruments also carry some unique risks.
* **Moderate risk (5-12%)**: Investment grade bonds 5-7%, large cap stocks 8-12%, real estate 7-10%. Still pretty safe.
* **Risky stuff (12%+)**: Small cap stocks 12-18%, private equity 15-25%, startups 25-50%, crypto 20-100%+ (good luck!). Hold onto your hat.

_The key principle_: investors want compensation for risk. There is no free lunch: If we accept more risk, we should expect higher returns.

<div>
    <center>
        <img src="figs/Fig-Bond-Asset-Timeline-Schematic.svg" width="580"/>
    </center>
</div>

## Abstract Assets and Net Cash Flows
An abstract asset is a sequence of dated cash flows denominated in a currency, such as dollars, euros, or yuan.

> __Abstract assets__ provide a general representation of the financial value associated with a process, product, or investment. Because cash flows at different dates have different units, they must be converted to a common valuation date before being added.

At cash-flow index $j$, let $\mathbf c_j\in\mathbb R_{\geq0}^{m}$ collect $m$ nonnegative cash-flow magnitudes and let $\boldsymbol\nu_j\in\{-1,+1\}^{m}$ specify their directions. A positive sign denotes an inflow and a negative sign denotes an outflow. The signed net cash flow at index $j$ is
$$
\begin{align*}
\bar c_j
=\left\langle\mathbf c_j,\boldsymbol\nu_j\right\rangle
=\sum_{i=1}^{m}c_{j,i}\nu_{j,i}.
\end{align*}
$$

Suppose the asset has cash-flow dates $j=0,1,\ldots,N$. The undiscounted nominal total
$$
\sum_{j=0}^{N}\bar c_j
$$
is not a time-0 value because its terms are denominated in dollars from different dates. Discounting each term to a common valuation date leads to net present value.


### Net Present Value (NPV)
Net present value is the sum of signed cash flows after every cash flow has been converted to a common valuation date. Let $\mathcal D_{j,0}^{-1}$ be the time-0 discount factor for cash-flow date $j$. Then
$$
\boxed{
\begin{align*}
\operatorname{NPV}_0
=\sum_{j=0}^{N}\mathcal D_{j,0}^{-1}\bar c_j.
\end{align*}}
$$

Under a constant nominal annual yield $y$ compounded $n$ times per year,
$$
\boxed{
\begin{align*}
\operatorname{NPV}_0(y)
&=\sum_{j=0}^{N}\mathcal D_{j,0}^{-1}(y;n)\bar c_j\\
&=\underbrace{\left\langle\mathcal D_{\star,0}^{-1}(y;n),\bar{\mathbf c}\right\rangle}_{\text{time-0 dollars}}.
\end{align*}}
$$
Every term in the sum is measured in time-0 dollars.

The sign of NPV compares the proposed cash flows with the declared discounting benchmark. It is a valuation statement, not a guarantee of accounting profit, realized return, or forecast accuracy.

* __Positive__: Discounted inflows exceed discounted outflows under the stated benchmark and assumptions.
* __Negative__: Discounted outflows exceed discounted inflows under the stated benchmark and assumptions.
* __Zero__: Discounted inflows equal discounted outflows. The transaction is fairly priced relative to that benchmark; zero NPV does not mean zero nominal gain or zero risk.

Let's apply this calculation in a worked example.

> __Example__
>
> [▶ Is buying a Tesla worth it?](./CHEME-5660-L1b-NetPresentValue-Fall-2026-WorkedExample.ipynb). This example constructs the dated ownership cash flows, discounts them to time $0$, and computes their NPV under a stated nominal-yield convention.


___

## Summary
This lecture introduced time-value-of-money calculations and the representation of investments as dated cash flows.

> __Key Takeaways:__
>
> * __Dated dollars must be compared at a common valuation date:__ A forward accumulation factor $\mathcal D$ moves value forward, while its inverse $\mathcal D^{-1}$ discounts future value to the valuation date.
> * __Rate conventions must be stated:__ A nominal annual yield $y$ compounded $n$ times per year has continuous equivalent $g_y=n\log(1+y/n)$; the dimensionless log return over an interval is $r=(\Delta t)g$.
> * __NPV values an abstract asset relative to a benchmark:__ Discount each signed cash flow $\bar c_j$ to time $0$ before adding. The sign of NPV depends on the cash-flow model and discount curve.

These ideas provide the valuation framework used throughout the course.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance.  Only risk capital that is not required for living expenses.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.